In [6]:
import pandas as pd

# 데이터 로드
residual1 = pd.read_csv('data/residual_features_frac_0.1.csv')
residual2 = pd.read_csv('data/residual_features_frac_0.2.csv')
residual3 = pd.read_csv('data/residual_features_frac_0.3.csv')
residual4 = pd.read_csv('data/residual_features_frac_0.4.csv')
residual5 = pd.read_csv('data/residual_features_frac_0.5.csv')
residual6 = pd.read_csv('data/residual_features_frac_0.6.csv')

In [7]:
from pct_covariance_shrinkage import PCTChart
import numpy as np

datasets = {
    0.1: residual1.copy(),
    0.2: residual2.copy(),
    0.3: residual3.copy(),
    0.4: residual4.copy(),
    0.5: residual5.copy(),
    0.6: residual6.copy(),
}

results_summary = []

for frac, df in datasets.items():
    feature_cols = [c for c in df.columns if c not in ["sample_idx", "status"]]

    X = df[feature_cols].copy()
    y = df["status"].copy()
    X = X.apply(pd.to_numeric, errors="coerce")

    train_mask = df["sample_idx"].between(0, 199)
    normal_mask = df["sample_idx"].between(200, 319)
    damaged_mask = df["sample_idx"].between(320, 399)

    X_train = X.loc[train_mask].to_numpy(dtype=float)
    X_normal = X.loc[normal_mask].to_numpy(dtype=float)
    X_damaged = X.loc[damaged_mask].to_numpy(dtype=float)

    for lam in [1.0, 0.9, 0.8, 0.6]:
        use_shrink = lam < 1.0

        pct = PCTChart(
            alpha=0.05,
            use_shrinkage=use_shrink,
            shrink_lambda=lam,
            shrink_target="trace",
            enforce_pd=True,
            pd_tol=1e-8,
        )
        pct.fit(X_train)

        res_normal = pct.score_samples(X_normal)
        res_damaged = pct.score_samples(X_damaged)

        far = np.mean(res_normal.alarms)
        dr = np.mean(res_damaged.alarms)

        results_summary.append({
            "frac": frac,
            "lambda": lam,
            "use_shrinkage": use_shrink,
            "FAR": far,
            "DR": dr,
            "mean_UCL_normal": np.nanmean(res_normal.ucls),
            "mean_Q_normal": np.nanmean(res_normal.q_values),
            "mean_Q_damaged": np.nanmean(res_damaged.q_values),
        })

summary_df = pd.DataFrame(results_summary)
summary_df = summary_df.sort_values(["frac", "lambda"]).reset_index(drop=True)
summary_df


,frac,lambda,use_shrinkage,FAR,DR,mean_UCL_normal,mean_Q_normal,mean_Q_damaged
0,0.1,0.6,True,0.050000,1.0000,2.249552,1.082356,24.000130
1,0.1,0.8,True,0.050000,1.0000,2.274181,1.082894,23.943931
2,0.1,0.9,True,0.050000,1.0000,2.288934,1.083243,23.917175
3,0.1,1.0,False,0.050000,1.0000,2.305253,1.083647,23.891313
4,0.2,0.6,True,0.083333,1.0000,2.307454,1.098380,21.634149
5,0.2,0.8,True,0.083333,1.0000,2.314835,1.101001,21.600316
6,0.2,0.9,True,0.083333,1.0000,2.319357,1.102665,21.587826
7,0.2,1.0,False,0.083333,1.0000,2.324444,1.104576,21.578320
8,0.3,0.6,True,0.075000,1.0000,2.421689,1.191468,21.345237
9,0.3,0.8,True,0.075000,1.0000,2.426951,1.195478,21.334428
